<a href="https://colab.research.google.com/github/ElynZeng/QM2-PROJECT/blob/main/Box_Plot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**3.Box Plot**

In [ ]:
#import libraries
#from google.colab import drive
#drive.mount('/content/drive')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# Read Japan's CSV
#df = pd.read_csv("drive/MyDrive/QM2 Group Project/Data/Japan_Data_cleaned.csv")
record_id = "18174921"
filename = "Japan_Data_cleaned.csv"
url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
df = pd.read_csv(url)

CITY_COL = "AREA"
YEAR_COL = "YEAR"
TEMP_COL = "TEMPERATURE"
GDP_COL  = "GDP PER CAPITA"

df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[TEMP_COL] = pd.to_numeric(df[TEMP_COL], errors="coerce")
df[GDP_COL]  = pd.to_numeric(df[GDP_COL], errors="coerce")
df = df.dropna(subset=[CITY_COL, YEAR_COL]).copy()

cities_all = sorted(df[CITY_COL].dropna().astype(str).unique().tolist())

# Setup Widgets
search_box = widgets.Text(
    value="",
    placeholder="Type to filter cities (e.g., Tokyo)...",
    description="Search:",
    layout=widgets.Layout(width="420px")
)

city_selector = widgets.SelectMultiple(
    options=cities_all,
    value=tuple(cities_all[:5]),
    description="Cities:",
    rows=12,
    layout=widgets.Layout(width="420px")
)

btn_select_all = widgets.Button(description="Select all", button_style="")
btn_clear = widgets.Button(description="Clear", button_style="")
out = widgets.Output()

def apply_filter(_=None):
    q = search_box.value.strip().lower()
    if not q:
        filtered = cities_all
    else:
        filtered = [c for c in cities_all if q in c.lower()]
    # preserve selected values if still visible
    current = set(city_selector.value)
    city_selector.options = filtered
    kept = tuple([c for c in filtered if c in current])
    city_selector.value = kept

def select_all(_):
    city_selector.value = tuple(city_selector.options)

def clear_all(_):
    city_selector.value = tuple()

btn_select_all.on_click(select_all)
btn_clear.on_click(clear_all)

def plot_compare_cities(selected_cities):
    sub = df[df[CITY_COL].astype(str).isin(list(selected_cities))].copy()
    cities = list(selected_cities)

    temp_data = [sub[sub[CITY_COL].astype(str) == c][TEMP_COL].dropna().values for c in cities]
    gdp_data  = [sub[sub[CITY_COL].astype(str) == c][GDP_COL].dropna().values for c in cities]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].boxplot(temp_data, tick_labels=cities, vert=True)
    axes[0].set_title(f"{TEMP_COL} by city (all years)")
    axes[0].set_ylabel("°C")
    axes[0].tick_params(axis="x", rotation=60)

    axes[1].boxplot(gdp_data, tick_labels=cities, vert=True)
    axes[1].set_title(f"{GDP_COL} by city (all years)")
    axes[1].set_ylabel("USD")
    axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    axes[1].tick_params(axis="x", rotation=60)

    plt.tight_layout()
    plt.show()

def refresh(*args):
    with out:
        out.clear_output(wait=True)
        if not city_selector.value:
            return
        plot_compare_cities(city_selector.value)

# events
search_box.observe(apply_filter, names="value")
city_selector.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

# initial display
controls = widgets.VBox([widgets.HBox([search_box, btn_select_all, btn_clear]),city_selector])
display(controls, out)

apply_filter()
refresh()

In [ ]:
#Read China's CSV
#df = pd.read_csv("drive/MyDrive/QM2 Group Project/Data/China_Data_cleaned.csv")
record_id = "18174921"
filename = "China_Data_cleaned.csv"
url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
df = pd.read_csv(url)


def pick_col(substrings):
    for c in df.columns:
        cl = c.lower()
        if any(s.lower() in cl for s in substrings):
            return c
    return None

YEAR_COL = pick_col(["year"])
TEMP_COL = pick_col(["temp", "temperature"])
GDP_COL  = pick_col(["gdp per capita", "gdp_per_capita", "gdppercapita", "per capita", "income", "gdp"])
CITY_COL = pick_col(["city", "province", "prefecture", "region", "area", "state", "name"])

YEAR_COL = "YEAR"
TEMP_COL = "TEMPERATURE"
GDP_COL  = "GDP PER CAPITA"
CITY_COL = "AREA"

# Clean Types
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[TEMP_COL] = pd.to_numeric(df[TEMP_COL], errors="coerce")
df[GDP_COL]  = pd.to_numeric(df[GDP_COL], errors="coerce")

df = df.dropna(subset=[CITY_COL, YEAR_COL]).copy()
cities_all = sorted(df[CITY_COL].dropna().astype(str).unique().tolist())

# Create Widgets
search_box = widgets.Text(
    value="",
    placeholder="Type to filter (e.g., Beijing)...",
    description="Search:",
    layout=widgets.Layout(width="420px")
)

city_selector = widgets.SelectMultiple(
    options=cities_all,
    value=tuple(cities_all[:5]),
    description="Cities:",
    rows=12,
    layout=widgets.Layout(width="420px")
)

btn_select_all = widgets.Button(description="Select all")
btn_clear = widgets.Button(description="Clear")
out = widgets.Output()

def apply_filter(_=None):
    q = search_box.value.strip().lower()
    if not q:
        filtered = cities_all
    else:
        filtered = [c for c in cities_all if q in c.lower()]

    current = set(city_selector.value)
    city_selector.options = filtered
    city_selector.value = tuple([c for c in filtered if c in current])

def select_all(_):
    city_selector.value = tuple(city_selector.options)

def clear_all(_):
    city_selector.value = tuple()

btn_select_all.on_click(select_all)
btn_clear.on_click(clear_all)

def plot_compare_cities(selected_cities):
    sub = df[df[CITY_COL].astype(str).isin(list(selected_cities))].copy()
    cities = list(selected_cities)

    # each city correspond to the array of values across all years
    temp_data = [sub[sub[CITY_COL].astype(str) == c][TEMP_COL].dropna().values for c in cities]
    gdp_data  = [sub[sub[CITY_COL].astype(str) == c][GDP_COL].dropna().values for c in cities]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].boxplot(temp_data, tick_labels=cities, vert=True)
    axes[0].set_title(f"{TEMP_COL} by {CITY_COL} (all years)")
    axes[0].set_ylabel("°C")
    axes[0].tick_params(axis="x", rotation=60)

    axes[1].boxplot(gdp_data, tick_labels=cities, vert=True)
    axes[1].set_title(f"{GDP_COL} by {CITY_COL} (all years)")
    axes[1].set_ylabel("USD")
    axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    axes[1].tick_params(axis="x", rotation=60)

    plt.tight_layout()
    plt.show()

def refresh(*args):
    with out:
        out.clear_output(wait=True)
        if not city_selector.value:
            return
        plot_compare_cities(city_selector.value)

# events
search_box.observe(apply_filter, names="value")
city_selector.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

# display
controls = widgets.VBox([widgets.HBox([search_box, btn_select_all, btn_clear]),city_selector])
display(controls, out)

apply_filter()
refresh()

In [ ]:
#Read Indonesia's CSV
#df = pd.read_csv("drive/MyDrive/QM2 Group Project/Data/Indonesia_Data_cleaned.csv")
record_id = "18174921"
filename = "Indonesia_Data_cleaned.csv"
url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
df = pd.read_csv(url)


def pick_col(substrings):
    for c in df.columns:
        cl = c.lower()
        if any(s.lower() in cl for s in substrings):
            return c
    return None

YEAR_COL = pick_col(["year"])
TEMP_COL = pick_col(["temp", "temperature"])
GDP_COL  = pick_col(["gdp per capita", "gdp_per_capita", "gdppercapita", "per capita", "income", "gdp"])
CITY_COL = pick_col(["city", "province", "region", "area", "state", "name"])

YEAR_COL = "YEAR"
TEMP_COL = "TEMPERATURE"
GDP_COL  = "GDP PER CAPITA"
CITY_COL = "AREA"

# Clean Types
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[TEMP_COL] = pd.to_numeric(df[TEMP_COL], errors="coerce")
df[GDP_COL]  = pd.to_numeric(df[GDP_COL], errors="coerce")

df = df.dropna(subset=[CITY_COL, YEAR_COL]).copy()
cities_all = sorted(df[CITY_COL].dropna().astype(str).unique().tolist())

# Create Widgets
search_box = widgets.Text(
    value="",
    placeholder="Type to filter (e.g., Jakarta)...",
    description="Search:",
    layout=widgets.Layout(width="420px")
)

city_selector = widgets.SelectMultiple(
    options=cities_all,
    value=tuple(cities_all[:5]),
    description="Cities:",
    rows=12,
    layout=widgets.Layout(width="420px")
)

btn_select_all = widgets.Button(description="Select all")
btn_clear = widgets.Button(description="Clear")
out = widgets.Output()

def apply_filter(_=None):
    q = search_box.value.strip().lower()
    if not q:
        filtered = cities_all
    else:
        filtered = [c for c in cities_all if q in c.lower()]

    current = set(city_selector.value)
    city_selector.options = filtered
    city_selector.value = tuple([c for c in filtered if c in current])

def select_all(_):
    city_selector.value = tuple(city_selector.options)

def clear_all(_):
    city_selector.value = tuple()

btn_select_all.on_click(select_all)
btn_clear.on_click(clear_all)

def plot_compare_locations(selected_locations):
    sub = df[df[CITY_COL].astype(str).isin(list(selected_locations))].copy()
    locs = list(selected_locations)

    temp_data = [sub[sub[CITY_COL].astype(str) == c][TEMP_COL].dropna().values for c in locs]
    gdp_data  = [sub[sub[CITY_COL].astype(str) == c][GDP_COL].dropna().values for c in locs]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].boxplot(temp_data, tick_labels=locs, vert=True)
    axes[0].set_title(f"{TEMP_COL} by {CITY_COL} (all years)")
    axes[0].set_ylabel("°C")
    axes[0].tick_params(axis="x", rotation=60)

    axes[1].boxplot(gdp_data, tick_labels=locs, vert=True)
    axes[1].set_title(f"{GDP_COL} by {CITY_COL} (all years)")
    axes[1].set_ylabel("USD")
    axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    axes[1].tick_params(axis="x", rotation=60)

    plt.tight_layout()
    plt.show()

def refresh(*args):
    with out:
        out.clear_output(wait=True)
        if not city_selector.value:
            return
        plot_compare_locations(city_selector.value)

# events
search_box.observe(apply_filter, names="value")
city_selector.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

# display
controls = widgets.VBox([widgets.HBox([search_box, btn_select_all, btn_clear]),city_selector])
display(controls, out)

apply_filter()
refresh()

In [ ]:
#Read South Korea's CSV
#df = pd.read_csv("drive/MyDrive/QM2 Group Project/Data/Korea_Data_cleaned.csv")
record_id = "18174921"
filename = "Korea_Data_cleaned.csv"
url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
df = pd.read_csv(url)


def pick_col(substrings):
    for c in df.columns:
        cl = c.lower()
        if any(s.lower() in cl for s in substrings):
            return c
    return None

YEAR_COL = pick_col(["year"])
TEMP_COL = pick_col(["temp", "temperature"])
GDP_COL  = pick_col(["gdp per capita", "gdp_per_capita", "gdppercapita", "per capita", "income", "gdp"])
CITY_COL = pick_col(["city", "province", "region", "area", "state", "name"])

YEAR_COL = "YEAR"
TEMP_COL = "TEMPERATURE"
GDP_COL  = "GDP PER CAPITA"
CITY_COL = "AREA"

df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[TEMP_COL] = pd.to_numeric(df[TEMP_COL], errors="coerce")
df[GDP_COL]  = pd.to_numeric(df[GDP_COL], errors="coerce")

df = df.dropna(subset=[CITY_COL, YEAR_COL]).copy()
cities_all = sorted(df[CITY_COL].dropna().astype(str).unique().tolist())

# Create Widgets
search_box = widgets.Text(
    value="",
    placeholder="Type to filter (e.g., Seoul)...",
    description="Search:",
    layout=widgets.Layout(width="420px")
)

city_selector = widgets.SelectMultiple(
    options=cities_all,
    value=tuple(cities_all[:5]),
    description="Cities:",
    rows=12,
    layout=widgets.Layout(width="420px")
)

btn_select_all = widgets.Button(description="Select all")
btn_clear = widgets.Button(description="Clear")
out = widgets.Output()

def apply_filter(_=None):
    q = search_box.value.strip().lower()
    if not q:
        filtered = cities_all
    else:
        filtered = [c for c in cities_all if q in c.lower()]

    current = set(city_selector.value)
    city_selector.options = filtered
    city_selector.value = tuple([c for c in filtered if c in current])

def select_all(_):
    city_selector.value = tuple(city_selector.options)

def clear_all(_):
    city_selector.value = tuple()

btn_select_all.on_click(select_all)
btn_clear.on_click(clear_all)

def plot_compare_locations(selected_locations):
    sub = df[df[CITY_COL].astype(str).isin(list(selected_locations))].copy()
    locs = list(selected_locations)

    temp_data = [sub[sub[CITY_COL].astype(str) == c][TEMP_COL].dropna().values for c in locs]
    gdp_data  = [sub[sub[CITY_COL].astype(str) == c][GDP_COL].dropna().values for c in locs]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].boxplot(temp_data, tick_labels=locs, vert=True)
    axes[0].set_title(f"{TEMP_COL} by {CITY_COL} (all years)")
    axes[0].set_ylabel("°C")
    axes[0].tick_params(axis="x", rotation=60)

    axes[1].boxplot(gdp_data, tick_labels=locs, vert=True)
    axes[1].set_title(f"{GDP_COL} by {CITY_COL} (all years)")
    axes[1].set_ylabel("USD")
    axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    axes[1].tick_params(axis="x", rotation=60)

    plt.tight_layout()
    plt.show()

def refresh(*args):
    with out:
        out.clear_output(wait=True)
        if not city_selector.value:
            return
        plot_compare_locations(city_selector.value)

# events
search_box.observe(apply_filter, names="value")
city_selector.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

# display
controls = widgets.VBox([widgets.HBox([search_box, btn_select_all, btn_clear]),city_selector])
display(controls, out)

apply_filter()
refresh()

In [ ]:
#Read Malaysia's CSV
#df = pd.read_csv("drive/MyDrive/QM2 Group Project/Data/Malaysia_Data_cleaned.csv")
record_id = "18174921"
filename = "Malaysia_Data_cleaned.csv"
url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
df = pd.read_csv(url)


def pick_col(substrings):
    for c in df.columns:
        cl = c.lower()
        if any(s.lower() in cl for s in substrings):
            return c
    return None

YEAR_COL = pick_col(["year"])
TEMP_COL = pick_col(["temp", "temperature"])
GDP_COL  = pick_col(["gdp per capita", "gdp_per_capita", "gdppercapita", "per capita", "income", "gdp"])
STATE_COL = pick_col(["state", "province", "region", "area", "name"])

YEAR_COL  = "YEAR"
TEMP_COL  = "TEMPERATURE"
GDP_COL   = "GDP PER CAPITA"
STATE_COL = "AREA"

# Clean Types
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[TEMP_COL] = pd.to_numeric(df[TEMP_COL], errors="coerce")
df[GDP_COL]  = pd.to_numeric(df[GDP_COL], errors="coerce")

df = df.dropna(subset=[STATE_COL, YEAR_COL]).copy()
states_all = sorted(df[STATE_COL].dropna().astype(str).unique().tolist())

# Create widgets
search_box = widgets.Text(
    value="",
    placeholder="Type to filter (e.g., Selangor)...",
    description="Search:",
    layout=widgets.Layout(width="420px")
)

state_selector = widgets.SelectMultiple(
    options=states_all,
    value=tuple(states_all[:5]),
    description="States:",
    rows=12,
    layout=widgets.Layout(width="420px")
)

btn_select_all = widgets.Button(description="Select all")
btn_clear = widgets.Button(description="Clear")
out = widgets.Output()

def apply_filter(_=None):
    q = search_box.value.strip().lower()
    if not q:
        filtered = states_all
    else:
        filtered = [s for s in states_all if q in s.lower()]

    current = set(state_selector.value)
    state_selector.options = filtered
    state_selector.value = tuple([s for s in filtered if s in current])

def select_all(_):
    state_selector.value = tuple(state_selector.options)

def clear_all(_):
    state_selector.value = tuple()

btn_select_all.on_click(select_all)
btn_clear.on_click(clear_all)

def plot_compare_states(selected_states):
    sub = df[df[STATE_COL].astype(str).isin(list(selected_states))].copy()
    states = list(selected_states)

    temp_data = [sub[sub[STATE_COL].astype(str) == s][TEMP_COL].dropna().values for s in states]
    gdp_data  = [sub[sub[STATE_COL].astype(str) == s][GDP_COL].dropna().values for s in states]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].boxplot(temp_data, tick_labels=states, vert=True)
    axes[0].set_title(f"{TEMP_COL} by state (all years)")
    axes[0].set_ylabel("°C")
    axes[0].tick_params(axis="x", rotation=60)

    axes[1].boxplot(gdp_data, tick_labels=states, vert=True)
    axes[1].set_title(f"{GDP_COL} by state (all years)")
    axes[1].set_ylabel("USD")
    axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    axes[1].tick_params(axis="x", rotation=60)

    plt.tight_layout()
    plt.show()

def refresh(*args):
    with out:
        out.clear_output(wait=True)
        if not state_selector.value:
            return
        plot_compare_states(state_selector.value)

# events
search_box.observe(apply_filter, names="value")
state_selector.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

# display
controls = widgets.VBox([widgets.HBox([search_box, btn_select_all, btn_clear]),state_selector])
display(controls, out)

apply_filter()
refresh()

In [ ]:
#Read Thailand's CSV
#df = pd.read_csv("drive/MyDrive/QM2 Group Project/Data/Thailand_Data_cleaned.csv")
record_id = "18174921"
filename = "Thailand_Data_cleaned.csv"
url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
df = pd.read_csv(url)


def pick_col(substrings):
    for c in df.columns:
        cl = c.lower()
        if any(s.lower() in cl for s in substrings):
            return c
    return None

# Detect Columns
YEAR_COL = pick_col(["year"])
TEMP_COL = pick_col(["temp", "temperature"])
GDP_COL  = pick_col(["gdp per capita", "gdp_per_capita", "gdppercapita", "per capita", "income", "gdp"])
LOC_COL = pick_col(["province", "city", "region", "area", "state", "name"])

YEAR_COL = "YEAR"
TEMP_COL = "TEMPERATURE"
GDP_COL  = "GDP PER CAPITA"
LOC_COL  = "AREA"

# Clean Types
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[TEMP_COL] = pd.to_numeric(df[TEMP_COL], errors="coerce")
df[GDP_COL]  = pd.to_numeric(df[GDP_COL], errors="coerce")

df = df.dropna(subset=[LOC_COL, YEAR_COL]).copy()
locs_all = sorted(df[LOC_COL].dropna().astype(str).unique().tolist())

# Create Widgets
search_box = widgets.Text(
    value="",
    placeholder="Type to filter (e.g., Bangkok)...",
    description="Search:",
    layout=widgets.Layout(width="420px")
)

loc_selector = widgets.SelectMultiple(
    options=locs_all,
    value=tuple(locs_all[:5]),
    description="Provinces:",
    rows=12,
    layout=widgets.Layout(width="420px")
)

btn_select_all = widgets.Button(description="Select all")
btn_clear = widgets.Button(description="Clear")
out = widgets.Output()

def apply_filter(_=None):
    q = search_box.value.strip().lower()
    if not q:
        filtered = locs_all
    else:
        filtered = [s for s in locs_all if q in s.lower()]

    current = set(loc_selector.value)
    loc_selector.options = filtered
    loc_selector.value = tuple([s for s in filtered if s in current])

def select_all(_):
    loc_selector.value = tuple(loc_selector.options)

def clear_all(_):
    loc_selector.value = tuple()

btn_select_all.on_click(select_all)
btn_clear.on_click(clear_all)

def plot_compare(selected_locs):
    sub = df[df[LOC_COL].astype(str).isin(list(selected_locs))].copy()
    locs = list(selected_locs)

    temp_data = [sub[sub[LOC_COL].astype(str) == s][TEMP_COL].dropna().values for s in locs]
    gdp_data  = [sub[sub[LOC_COL].astype(str) == s][GDP_COL].dropna().values for s in locs]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].boxplot(temp_data, tick_labels=locs, vert=True)
    axes[0].set_title(f"{TEMP_COL} by {LOC_COL} (all years)")
    axes[0].set_ylabel("°C")
    axes[0].tick_params(axis="x", rotation=60)

    axes[1].boxplot(gdp_data, tick_labels=locs, vert=True)
    axes[1].set_title(f"{GDP_COL} by {LOC_COL} (all years)")
    axes[1].set_ylabel("USD")
    axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    axes[1].tick_params(axis="x", rotation=60)

    plt.tight_layout()
    plt.show()

def refresh(*args):
    with out:
        out.clear_output(wait=True)
        if not loc_selector.value:
            return
        plot_compare(loc_selector.value)

# events
search_box.observe(apply_filter, names="value")
loc_selector.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

# display
controls = widgets.VBox([widgets.HBox([search_box, btn_select_all, btn_clear]),loc_selector])
display(controls, out)

apply_filter()
refresh()

In [ ]:
#Read Vietnam's CSV
#df = pd.read_csv("drive/MyDrive/QM2 Group Project/Data/Vietnam_Data_cleaned.csv")
record_id = "18174921"
filename = "Vietnam_Data_cleaned.csv"
url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
df = pd.read_csv(url)


def pick_col(substrings):
    for c in df.columns:
        cl = c.lower()
        if any(s.lower() in cl for s in substrings):
            return c
    return None

YEAR_COL = pick_col(["year"])
TEMP_COL = pick_col(["temp", "temperature"])
GDP_COL  = pick_col(["gdp per capita", "gdp_per_capita", "gdppercapita", "per capita", "income", "gdp"])
LOC_COL = pick_col(["province", "city", "region", "area", "state", "name"])

YEAR_COL = "YEAR"
TEMP_COL = "TEMPERATURE"
GDP_COL  = "GDP PER CAPITA"
LOC_COL  = "AREA"

# Clean Types
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[TEMP_COL] = pd.to_numeric(df[TEMP_COL], errors="coerce")
df[GDP_COL]  = pd.to_numeric(df[GDP_COL], errors="coerce")

df = df.dropna(subset=[LOC_COL, YEAR_COL]).copy()
locs_all = sorted(df[LOC_COL].dropna().astype(str).unique().tolist())

# Create Widgets
search_box = widgets.Text(
    value="",
    placeholder="Type to filter (e.g., Hanoi)...",
    description="Search:",
    layout=widgets.Layout(width="420px")
)

loc_selector = widgets.SelectMultiple(
    options=locs_all,
    value=tuple(locs_all[:5]),
    description="Provinces:",
    rows=12,
    layout=widgets.Layout(width="420px")
)

btn_select_all = widgets.Button(description="Select all")
btn_clear = widgets.Button(description="Clear")
out = widgets.Output()

def apply_filter(_=None):
    q = search_box.value.strip().lower()
    if not q:
        filtered = locs_all
    else:
        filtered = [s for s in locs_all if q in s.lower()]

    current = set(loc_selector.value)
    loc_selector.options = filtered
    loc_selector.value = tuple([s for s in filtered if s in current])

def select_all(_):
    loc_selector.value = tuple(loc_selector.options)

def clear_all(_):
    loc_selector.value = tuple()

btn_select_all.on_click(select_all)
btn_clear.on_click(clear_all)

def plot_compare(selected_locs):
    sub = df[df[LOC_COL].astype(str).isin(list(selected_locs))].copy()
    locs = list(selected_locs)

    temp_data = [sub[sub[LOC_COL].astype(str) == s][TEMP_COL].dropna().values for s in locs]
    gdp_data  = [sub[sub[LOC_COL].astype(str) == s][GDP_COL].dropna().values for s in locs]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].boxplot(temp_data, tick_labels=locs, vert=True)
    axes[0].set_title(f"{TEMP_COL} by {LOC_COL} (all years)")
    axes[0].set_ylabel("°C")
    axes[0].tick_params(axis="x", rotation=60)

    axes[1].boxplot(gdp_data, tick_labels=locs, vert=True)
    axes[1].set_title(f"{GDP_COL} by {LOC_COL} (all years)")
    axes[1].set_ylabel("USD")
    axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    axes[1].tick_params(axis="x", rotation=60)

    plt.tight_layout()
    plt.show()

def refresh(*args):
    with out:
        out.clear_output(wait=True)
        if not loc_selector.value:
            return
        plot_compare(loc_selector.value)

# events
search_box.observe(apply_filter, names="value")
loc_selector.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

# display
controls = widgets.VBox([widgets.HBox([search_box, btn_select_all, btn_clear]),loc_selector])
display(controls, out)

apply_filter()
refresh()